<a href="https://colab.research.google.com/github/vika3mm/gost_progect/blob/main/%D1%81%D0%B1%D0%BE%D1%80%D0%BA%D0%B0_tex.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
!pip install -q transformers pymupdf pillow numpy

In [21]:
import os
import re
import fitz
import torch
import numpy as np

from PIL import Image
from collections import Counter
from google.colab import drive

from transformers import (
    LayoutLMv3Processor,
    LayoutLMv3ForTokenClassification
)

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [22]:
MODEL_PATH = (
    "/content/drive/MyDrive/GOST_Project/"
    "layoutlm_model_split"
)

INPUT_PDF = (
    "/content/drive/MyDrive/GOST_Project/"
    "Денисова_ВКР_поломанный.pdf"
)

OUTPUT_TEX = (
    "/content/drive/MyDrive/GOST_Project/"
    "smart_result_latex.tex"
)

print("PDF:", INPUT_PDF)
print("MODEL:", MODEL_PATH)
print("OUTPUT:", OUTPUT_TEX)

PDF: /content/drive/MyDrive/GOST_Project/Денисова_ВКР_поломанный.pdf
MODEL: /content/drive/MyDrive/GOST_Project/layoutlm_model_split
OUTPUT: /content/drive/MyDrive/GOST_Project/smart_result_latex.tex


In [23]:
device = torch.device(
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)

print("Device:", device)

processor = LayoutLMv3Processor.from_pretrained(
    "microsoft/layoutlmv3-base",
    apply_ocr=False
)

model = LayoutLMv3ForTokenClassification.from_pretrained(
    MODEL_PATH
)

model.to(device)
model.eval()

id2label = {
    0: "BODY",
    1: "FIGURE",
    2: "FIGURE_CAPTION",
    3: "FOOTNOTE",
    4: "FORMULA",
    5: "OTHER",
    6: "REFERENCE",
    7: "SECTION_HEADER",
    8: "TABLE",
    9: "TABLE_CAPTION",
    10: "TITLE",
    11: "TOC",
}

print("Модель загружена.")

Device: cpu


Loading weights:   0%|          | 0/216 [00:00<?, ?it/s]

Модель загружена.


In [24]:
def extract_pymu_data(pdf_path):
    """
    Извлекает слова и bbox из PDF через PyMuPDF.
    bbox нормализуются в диапазон 0..1000.
    """

    doc = fitz.open(pdf_path)

    all_pages = {}

    for page_index in range(len(doc)):

        page = doc[page_index]

        words_data = page.get_text(
            "words",
            sort=True
        )

        page_width = page.rect.width
        page_height = page.rect.height

        words = []

        for word_index, w in enumerate(
            words_data
        ):

            x0, y0, x1, y1 = (
                w[0],
                w[1],
                w[2],
                w[3]
            )

            text = str(w[4])

            block_no = int(w[5])
            line_no = int(w[6])
            word_no = int(w[7])

            box = [
                max(
                    0,
                    min(
                        1000,
                        int(
                            x0
                            / page_width
                            * 1000
                        )
                    )
                ),
                max(
                    0,
                    min(
                        1000,
                        int(
                            y0
                            / page_height
                            * 1000
                        )
                    )
                ),
                max(
                    0,
                    min(
                        1000,
                        int(
                            x1
                            / page_width
                            * 1000
                        )
                    )
                ),
                max(
                    0,
                    min(
                        1000,
                        int(
                            y1
                            / page_height
                            * 1000
                        )
                    )
                ),
            ]

            words.append({
                "text": text,
                "box": box,
                "block_no": block_no,
                "line_no": line_no,
                "word_no": word_no,
                "word_index": word_index,
            })

        all_pages[
            page_index + 1
        ] = {
            "words": words,
            "width": page_width,
            "height": page_height,
        }

    doc.close()

    return all_pages


print(
    "📄 Извлекаем текст "
    "из PDF..."
)

pymu_data = extract_pymu_data(
    INPUT_PDF
)

print(
    f"✅ Извлечено "
    f"{len(pymu_data)} страниц"
)

📄 Извлекаем текст из PDF...
✅ Извлечено 39 страниц


In [25]:
def predict_page(
    word_items,
    max_words=50
):

    if not word_items:
        return []

    valid_items = [
        item
        for item in word_items
        if item["text"].strip()
    ]

    all_predictions = []

    for start in range(
        0,
        len(valid_items),
        max_words
    ):

        chunk = valid_items[
            start:start + max_words
        ]

        chunk_words = [
            item["text"]
            for item in chunk
        ]

        chunk_boxes = [
            item["box"]
            for item in chunk
        ]

        # Как и в вашей текущей
        # inference-версии.
        image = Image.new(
            "RGB",
            (1000, 1000),
            color="white"
        )

        encoding = processor(
            image,
            chunk_words,
            boxes=chunk_boxes,
            truncation=True,
            padding="max_length",
            max_length=512,
            return_tensors="pt"
        )

        word_ids = encoding.word_ids(
            batch_index=0
        )

        model_inputs = {
            key: value.to(device)
            for key, value
            in encoding.items()
        }

        with torch.no_grad():

            outputs = model(
                **model_inputs
            )

        preds = (
            outputs.logits
            .argmax(dim=-1)[0]
            .cpu()
        )

        used_word_ids = set()

        for token_index, word_id in enumerate(
            word_ids
        ):

            if word_id is None:
                continue

            if word_id in used_word_ids:
                continue

            if word_id >= len(chunk):
                continue

            used_word_ids.add(
                word_id
            )

            original_item = chunk[
                word_id
            ]

            label_id = int(
                preds[
                    token_index
                ].item()
            )

            all_predictions.append({
                **original_item,
                "label": id2label[
                    label_id
                ],
            })

    all_predictions.sort(
        key=lambda x:
        x["word_index"]
    )

    return all_predictions

In [26]:
def build_lines(predictions):

    if not predictions:
        return []

    items = []

    for item in predictions:

        text = item[
            "text"
        ].strip()

        if not text:
            continue

        x0, y0, x1, y1 = (
            item["box"]
        )

        item = dict(item)

        item["_cy"] = (
            y0 + y1
        ) / 2

        item["_height"] = max(
            1,
            y1 - y0
        )

        items.append(item)

    if not items:
        return []

    items.sort(
        key=lambda x: (
            x["_cy"],
            x["box"][0]
        )
    )

    visual_lines = []

    for item in items:

        x0, y0, x1, y1 = (
            item["box"]
        )

        cy = item["_cy"]
        height = item["_height"]

        best_line = None
        best_distance = None

        for line in reversed(
            visual_lines[-10:]
        ):

            line_y0 = line["y0"]
            line_y1 = line["y1"]
            line_cy = line["cy"]

            line_height = max(
                1,
                line_y1 - line_y0
            )

            overlap = max(
                0,
                min(
                    y1,
                    line_y1
                )
                -
                max(
                    y0,
                    line_y0
                )
            )

            overlap_ratio = (
                overlap
                /
                max(
                    1,
                    min(
                        height,
                        line_height
                    )
                )
            )

            center_distance = abs(
                cy - line_cy
            )

            same_visual_line = (
                overlap_ratio >= 0.35
                or
                center_distance
                <=
                max(
                    4,
                    min(
                        height,
                        line_height
                    ) * 0.45
                )
            )

            if not same_visual_line:
                continue

            if (
                best_distance is None
                or
                center_distance
                < best_distance
            ):
                best_line = line
                best_distance = (
                    center_distance
                )

        if best_line is None:

            visual_lines.append({
                "items": [item],
                "y0": y0,
                "y1": y1,
                "cy": cy,
            })

        else:

            best_line[
                "items"
            ].append(item)

            best_line["y0"] = min(
                best_line["y0"],
                y0
            )

            best_line["y1"] = max(
                best_line["y1"],
                y1
            )

            best_line["cy"] = (
                sum(
                    x["_cy"]
                    for x
                    in best_line[
                        "items"
                    ]
                )
                /
                len(
                    best_line[
                        "items"
                    ]
                )
            )

    result = []

    for visual_line in visual_lines:

        line_items = sorted(
            visual_line["items"],
            key=lambda x:
            x["box"][0]
        )

        text = " ".join(
            x["text"]
            for x in line_items
        ).strip()

        if not text:
            continue

        labels = [
            x["label"]
            for x in line_items
        ]

        counts = Counter(
            labels
        )

        main_label = (
            counts
            .most_common(1)[0][0]
        )

        ref_count = counts.get(
            "REFERENCE",
            0
        )

        toc_count = counts.get(
            "TOC",
            0
        )

        if (
            ref_count > 0
            and
            ref_count
            >=
            len(line_items) * 0.5
        ):

            main_label = "REFERENCE"

        elif (
            toc_count > 0
            and
            toc_count
            >=
            len(line_items) * 0.5
        ):

            main_label = "TOC"

        result.append({
            "text": text,
            "label": main_label,
            "box": [
                min(
                    x["box"][0]
                    for x
                    in line_items
                ),
                min(
                    x["box"][1]
                    for x
                    in line_items
                ),
                max(
                    x["box"][2]
                    for x
                    in line_items
                ),
                max(
                    x["box"][3]
                    for x
                    in line_items
                ),
            ],
            "word_labels": labels,
        })

    result.sort(
        key=lambda x: (
            x["box"][1],
            x["box"][0]
        )
    )

    return result

In [27]:
print(
    "🔄 Классифицируем "
    "все страницы..."
)

all_pages_lines = {}

for page_num in sorted(
    pymu_data.keys()
):

    word_items = (
        pymu_data[
            page_num
        ]["words"]
    )

    predictions = predict_page(
        word_items,
        max_words=50
    )

    lines = build_lines(
        predictions
    )

    for line in lines:
        line["page"] = page_num

    all_pages_lines[
        page_num
    ] = lines


print(
    f"✅ Обработано "
    f"{len(all_pages_lines)} "
    f"страниц"
)

🔄 Классифицируем все страницы...
✅ Обработано 39 страниц


In [28]:
def find_page_by_label_density(
    all_pages_lines,
    label
):

    page_scores = {}

    for page_num, lines in (
        all_pages_lines.items()
    ):

        if not lines:
            continue

        label_count = sum(
            1
            for line in lines
            if line["label"] == label
        )

        if label_count == 0:
            continue

        ratio = (
            label_count
            / len(lines)
        )

        page_scores[
            page_num
        ] = {
            "count": label_count,
            "ratio": ratio,
        }

    if not page_scores:
        return None

    best_page = max(
        page_scores,
        key=lambda p: (
            page_scores[p]["count"],
            page_scores[p]["ratio"]
        )
    )

    score = page_scores[
        best_page
    ]

    print(
        f"📄 Страница {best_page}: "
        f"{score['count']} строк "
        f"{label}, "
        f"плотность "
        f"{score['ratio']:.2f}"
    )

    return best_page


print(
    "\n🔍 Ищем страницы..."
)

toc_page = (
    find_page_by_label_density(
        all_pages_lines,
        "TOC"
    )
)

ref_page = (
    find_page_by_label_density(
        all_pages_lines,
        "REFERENCE"
    )
)

print(
    "Страница содержания:",
    toc_page
)

print(
    "Страница библиографии:",
    ref_page
)


🔍 Ищем страницы...
📄 Страница 2: 20 строк TOC, плотность 0.91
📄 Страница 37: 28 строк REFERENCE, плотность 0.97
Страница содержания: 2
Страница библиографии: 37


In [29]:
def extract_toc(
    all_pages_lines,
    toc_page
):

    if toc_page is None:
        return []

    page_stats = {}

    for page_num, lines in (
        all_pages_lines.items()
    ):

        if not lines:
            continue

        toc_count = sum(
            1
            for line in lines
            if line["label"] == "TOC"
        )

        toc_ratio = (
            toc_count
            / len(lines)
        )

        page_stats[
            page_num
        ] = {
            "toc_count": toc_count,
            "toc_ratio": toc_ratio,
        }

    def is_toc_page(page_num):

        if page_num not in page_stats:
            return False

        stats = page_stats[
            page_num
        ]

        return (
            stats[
                "toc_count"
            ] >= 3
            or
            stats[
                "toc_ratio"
            ] >= 0.25
        )

    selected_pages = [
        toc_page
    ]

    page = toc_page - 1

    while (
        page in all_pages_lines
        and
        is_toc_page(page)
    ):
        selected_pages.insert(
            0,
            page
        )

        page -= 1

    page = toc_page + 1

    while (
        page in all_pages_lines
        and
        is_toc_page(page)
    ):
        selected_pages.append(
            page
        )

        page += 1

    print(
        "📑 Страницы содержания:",
        ", ".join(
            map(
                str,
                selected_pages
            )
        )
    )

    result = []

    for page_num in selected_pages:

        for line in (
            all_pages_lines[
                page_num
            ]
        ):

            if (
                line["label"]
                != "TOC"
            ):
                continue

            text = (
                line["text"]
                .strip()
            )

            if not text:
                continue

            result.append({
                **line,
                "page": page_num,
            })

    result.sort(
        key=lambda x: (
            x["page"],
            x["box"][1],
            x["box"][0]
        )
    )

    # Убираем сам заголовок
    # "Содержание",
    # поскольку LaTeX создаст его
    # самостоятельно.
    if result:

        first_text = (
            result[0]["text"]
            .strip()
            .lower()
        )

        if first_text in (
            "содержание",
            "оглавление"
        ):
            result = result[1:]

    return result


print(
    "\n📖 Извлекаем содержание..."
)

toc_lines = extract_toc(
    all_pages_lines,
    toc_page
)

print(
    "✅ TOC строк:",
    len(toc_lines)
)


📖 Извлекаем содержание...
📑 Страницы содержания: 2
✅ TOC строк: 20


In [30]:
def normalize_reference_text(text):

    if text is None:
        return ""

    text = str(text)

    text = text.replace(
        "\u00a0",
        " "
    )

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def is_ref_heading(text):

    return (
        text
        .strip()
        .lower()
        in [
            "список литературы",
            "библиографический список",
            "библиография",
            "список использованных источников",
        ]
    )


def is_page_number_line(line):

    text = normalize_reference_text(
        line["text"]
    )

    if not re.fullmatch(
        r"\d{1,4}",
        text
    ):
        return False

    y0 = line["box"][1]
    y1 = line["box"][3]

    return (
        y0 <= 100
        or
        y1 >= 900
    )


YEAR_RE = re.compile(
    r"\("
    r"(?:"
    r"n\.d\.(?:-[a-z])?"
    r"|"
    r"(?:18|19|20)\d{2}"
    r"(?:[^)]{0,40})?"
    r")"
    r"\)",
    flags=re.IGNORECASE
)


def contains_year(text):

    return (
        YEAR_RE.search(
            normalize_reference_text(
                text
            )
        )
        is not None
    )


def starts_with_year(text):

    text = normalize_reference_text(
        text
    )

    return (
        re.match(
            r"^\s*(?:"
            r"\((?:18|19|20)"
            r"\d{2}[^)]*\)"
            r"|"
            r"(?:18|19|20)\d{2}"
            r")"
            r"[\s.,:]",
            text
        )
        is not None
    )

In [31]:
def starts_with_author(text):

    text = normalize_reference_text(
        text
    )

    if not text:
        return False

    if re.match(
        r"^(?:DOI|URL|ISBN|ISSN)\b",
        text,
        flags=re.IGNORECASE
    ):
        return False

    pattern1 = (
        r"^[A-ZА-ЯЁ]"
        r"[A-Za-zА-Яа-яЁё'’\-]+"
        r"(?:\s+[A-ZА-ЯЁ]\.){1,4}"
        r"(?:\s|$)"
    )

    if re.match(
        pattern1,
        text
    ):
        return True

    pattern2 = (
        r"^[A-ZА-ЯЁ]"
        r"[A-Za-zА-Яа-яЁё'’\-]+"
        r"(?:\s+"
        r"[A-Za-zА-Яа-яЁё'’\-]+)"
        r"{0,3}"
        r",\s*"
        r"(?:[A-ZА-ЯЁ]\.){1,4}"
    )

    return (
        re.match(
            pattern2,
            text
        )
        is not None
    )


def looks_like_full_name_authors(
    text
):

    text = normalize_reference_text(
        text
    )

    if not re.search(
        r"\b(?:and|&)\b",
        text,
        flags=re.IGNORECASE
    ):
        return False

    bad_words = {
        "software",
        "testing",
        "security",
        "privacy",
        "analysis",
        "system",
        "systems",
        "generation",
        "techniques",
        "implications",
        "symposium",
        "conference",
        "proceedings",
        "information",
        "construction",
        "exploit",
    }

    words = re.findall(
        r"[A-Za-zА-Яа-яЁё'’\-]+",
        text
    )

    lower = {
        w.lower()
        for w in words
    }

    if lower & bad_words:
        return False

    capitalized = [
        w
        for w in words
        if (
            w
            and
            w[0].isupper()
        )
    ]

    return (
        len(capitalized) >= 5
    )


def forced_continuation(
    previous,
    current
):

    previous = normalize_reference_text(
        previous
    )

    current = normalize_reference_text(
        current
    )

    if re.match(
        r"^(?:DOI|URL|ISBN|ISSN)\s*:",
        current,
        flags=re.IGNORECASE
    ):
        return True

    if re.match(
        r"^https?://",
        current,
        flags=re.IGNORECASE
    ):
        return True

    if (
        re.search(
            r"\d[\-–—]$",
            previous
        )
        and
        re.match(
            r"^\d",
            current
        )
    ):
        return True

    if re.search(
        r"[A-Za-zА-Яа-яЁё]-$",
        previous
    ):
        return True

    if re.search(
        r"(?:&|\band)\s*$",
        previous,
        flags=re.IGNORECASE
    ):
        return True

    return False

In [32]:
def is_new_reference(
    lines,
    i
):

    current = normalize_reference_text(
        lines[i]["text"]
    )

    if i == 0:
        return True

    previous = normalize_reference_text(
        lines[i - 1]["text"]
    )

    next_text = ""

    if i + 1 < len(lines):

        next_text = normalize_reference_text(
            lines[
                i + 1
            ]["text"]
        )

    if forced_continuation(
        previous,
        current
    ):
        return False

    # Продолжение диапазона
    # или строки, начинающейся
    # с номера.
    if re.match(
        r"^\d",
        current
    ):
        return False

    if starts_with_year(
        current
    ):
        return False

    if (
        starts_with_author(
            current
        )
        and
        contains_year(
            current
        )
    ):
        return True

    if starts_with_author(
        current
    ):

        if starts_with_year(
            next_text
        ):
            return True

        if (
            starts_with_author(
                next_text
            )
            and
            contains_year(
                next_text
            )
        ):
            return True

        return False

    # Организация / электронный
    # ресурс и год в той же строке.
    if (
        re.match(
            r"^[A-ZА-ЯЁ]",
            current
        )
        and
        contains_year(
            current
        )
    ):
        return True

    if (
        looks_like_full_name_authors(
            current
        )
        and
        starts_with_year(
            next_text
        )
    ):
        return True

    return False


def join_reference_lines(
    left,
    right
):

    left = normalize_reference_text(
        left
    )

    right = normalize_reference_text(
        right
    )

    if not left:
        return right

    if not right:
        return left

    # 74- + 84
    if (
        re.search(
            r"\d[\-–—]$",
            left
        )
        and
        re.match(
            r"^\d",
            right
        )
    ):
        return left + right

    # Разорванный URL
    if re.search(
        r"https?://\S+$",
        left,
        flags=re.IGNORECASE
    ):

        if (
            len(right) <= 15
            and
            re.fullmatch(
                r"[A-Za-z0-9."
                r"_~/%?#=&+\-]+",
                right
            )
        ):
            return left + right

    # Сохраняем дефис:
    # Patch- + Based
    if re.search(
        r"[A-Za-zА-Яа-яЁё]-$",
        left
    ):
        return left + right

    return (
        left
        + " "
        + right
    )
def starts_with_explicit_reference_number(text):
    """
    Явная нумерация библиографии:
    1. ...
    2) ...
    [3] ...
    """

    text = normalize_reference_text(text)

    return re.match(
        r"^\s*(?:\[\d+\]|\d+[.)])\s+",
        text
    ) is not None


def get_reference_gap_thresholds(lines):
    """
    Автоматически оценивает вертикальный промежуток,
    характерный для начала нового библиографического
    источника.

    Порог вычисляется отдельно для каждой страницы.
    """

    page_gaps = {}

    for i in range(1, len(lines)):

        prev = lines[i - 1]
        curr = lines[i]

        prev_page = prev.get("page")
        curr_page = curr.get("page")

        # Геометрические координаты разных страниц
        # сравнивать нельзя.
        if prev_page != curr_page:
            continue

        gap = (
            curr["box"][1]
            - prev["box"][3]
        )

        if gap <= 0:
            continue

        page_gaps.setdefault(
            curr_page,
            []
        ).append(gap)

    thresholds = {}

    for page_num, gaps in page_gaps.items():

        if not gaps:
            continue

        gaps = sorted(gaps)

        median_gap = float(
            np.median(gaps)
        )

        deviations = [
            abs(g - median_gap)
            for g in gaps
        ]

        mad = float(
            np.median(deviations)
        )

        threshold = max(
            median_gap * 2.2,
            median_gap + max(
                8.0,
                mad * 4.0
            )
        )

        thresholds[page_num] = (
            threshold
        )

    return thresholds


def has_strong_paragraph_gap(
    previous_line,
    current_line,
    thresholds
):
    """
    Проверяет, есть ли между строками выраженный
    вертикальный интервал — признак нового источника.
    """

    prev_page = previous_line.get(
        "page"
    )

    curr_page = current_line.get(
        "page"
    )

    if prev_page != curr_page:
        return False

    if curr_page not in thresholds:
        return False

    gap = (
        current_line["box"][1]
        - previous_line["box"][3]
    )

    return (
        gap
        >= thresholds[curr_page]
    )


def is_author_list_continuation(
    previous_text,
    current_text
):
    """
    Не позволяет разрезать длинный список авторов.

    Например:

    Huang, ..., Simon, I.,
    Hawthorne, ..., Eck, D. (2018).

    Это одна библиографическая запись.
    """

    previous_text = (
        normalize_reference_text(
            previous_text
        )
    )

    current_text = (
        normalize_reference_text(
            current_text
        )
    )

    if (
        not previous_text
        or not current_text
    ):
        return False

    # Предыдущая физическая строка закончилась
    # посреди перечисления авторов.
    if re.search(
        r",\s*$",
        previous_text
    ):

        if re.match(
            r"^(?:"
            r"[A-ZА-ЯЁ]"
            r"[A-Za-zА-Яа-яЁё'’\-]+"
            r"|"
            r"[A-ZА-ЯЁ]\."
            r")",
            current_text
        ):
            return True

    # Перенос сразу после & / and.
    if re.search(
        r"(?:&|\band)\s*$",
        previous_text,
        flags=re.IGNORECASE
    ):
        return True

    return False

def group_references(lines):
    """
    Восстанавливает логические библиографические записи.

    Использует:
    - геометрию документа;
    - явную нумерацию;
    - защиту длинных списков авторов;
    - старые текстовые эвристики.
    """

    if not lines:
        return []

    lines = sorted(
        lines,
        key=lambda x: (
            x.get("page", 0),
            x["box"][1],
            x["box"][0]
        )
    )

    gap_thresholds = (
        get_reference_gap_thresholds(
            lines
        )
    )

    print(
        "\n📐 Автоматические пороги "
        "абзацных интервалов:"
    )

    for page_num in sorted(
        gap_thresholds
    ):

        print(
            f"   page {page_num}: "
            f"{gap_thresholds[page_num]:.1f}"
        )

    records = []

    current = ""

    for i, line in enumerate(lines):

        text = normalize_reference_text(
            line["text"]
        )

        if not text:
            continue

        # Самая первая строка
        if i == 0:

            current = text
            continue

        previous_line = (
            lines[i - 1]
        )

        previous_text = (
            normalize_reference_text(
                previous_line["text"]
            )
        )

        # 1. Визуальный разрыв
        paragraph_break = (
            has_strong_paragraph_gap(
                previous_line,
                line,
                gap_thresholds
            )
        )

        # 2. Явный номер записи
        explicit_number = (
            starts_with_explicit_reference_number(
                text
            )
        )

        # 3. Продолжение списка авторов
        author_continuation = (
            is_author_list_continuation(
                previous_text,
                text
            )
        )

        # 4. Старая текстовая логика
        semantic_start = (
            is_new_reference(
                lines,
                i
            )
        )

        new_reference = False

        if paragraph_break:

            new_reference = True

        elif explicit_number:

            new_reference = True

        elif author_continuation:

            new_reference = False

        elif semantic_start:

            new_reference = True

        if new_reference:

            if current:

                records.append(
                    normalize_reference_text(
                        current
                    )
                )

            current = text

        else:

            current = (
                join_reference_lines(
                    current,
                    text
                )
            )

    if current:

        records.append(
            normalize_reference_text(
                current
            )
        )

    return records

In [33]:
def extract_references(
    all_pages_lines,
    ref_page
):

    if ref_page is None:
        return []

    reference_pages = []

    for page_num, lines in (
        all_pages_lines.items()
    ):

        if any(
            line["label"]
            == "REFERENCE"
            for line in lines
        ):

            reference_pages.append(
                page_num
            )

    reference_pages.sort()

    if not reference_pages:
        return []

    # Непрерывный блок
    # вокруг основной страницы.
    selected_pages = [
        ref_page
    ]

    p = ref_page - 1

    while p in reference_pages:

        selected_pages.insert(
            0,
            p
        )

        p -= 1

    p = ref_page + 1

    while p in reference_pages:

        selected_pages.append(
            p
        )

        p += 1

    print(
        "📚 Страницы библиографии:",
        ", ".join(
            map(
                str,
                selected_pages
            )
        )
    )

    reference_lines = []

    for page_num in selected_pages:

        page_lines = (
            all_pages_lines[
                page_num
            ]
        )

        # ВАЖНО:
        # здесь больше НЕТ
        #
        # if line["label"] !=
        #    "REFERENCE": continue
        #
        # потому что единичная
        # ошибка classifier не должна
        # удалять кусок источника.

        for line in page_lines:

            text = normalize_reference_text(
                line["text"]
            )

            if not text:
                continue

            if is_ref_heading(
                text
            ):
                continue

            if is_page_number_line(
                line
            ):
                continue

            item = dict(line)

            item["page"] = (
                page_num
            )

            item["text"] = text

            reference_lines.append(
                item
            )

    reference_lines.sort(
        key=lambda x: (
            x["page"],
            x["box"][1],
            x["box"][0]
        )
    )

    print(
        "🧩 Физических строк:",
        len(reference_lines)
    )

    records = group_references(
        reference_lines
    )

    records = [
        x
        for x in records
        if len(x) >= 8
    ]

    print(
        "✅ Восстановлено "
        "записей:",
        len(records)
    )

    return records


print(
    "\n📚 Извлекаем "
    "библиографию..."
)

ref_lines = extract_references(
    all_pages_lines,
    ref_page
)

print(
    "✅ References:",
    len(ref_lines)
)


📚 Извлекаем библиографию...
📚 Страницы библиографии: 37, 38, 39
🧩 Физических строк: 72

📐 Автоматические пороги абзацных интервалов:
   page 37: 11.0
   page 38: 11.0
   page 39: 11.0
✅ Восстановлено записей: 23
✅ References: 23


In [34]:
print(
    "\n"
    + "=" * 80
)

print(
    "СОДЕРЖАНИЕ"
)

print(
    "=" * 80
)

for item in toc_lines:
    print(
        item["text"]
    )


print(
    "\n"
    + "=" * 80
)

print(
    "СПИСОК ЛИТЕРАТУРЫ"
)

print(
    "=" * 80
)

for i, entry in enumerate(
    ref_lines,
    start=1
):
    print(
        f"{i}. {entry}"
    )


СОДЕРЖАНИЕ
СПИСОК ТЕРМИНОВ И АББРЕВИАТУР 2
ВВЕДЕНИЕ 3
1. ОБЗОР ЛИТЕРАТУРЫ 5
1.1. Введение в генерацию музыки 5
1.2. Существующие подходы к генерации символьной музыки 7
1.3. Представления MIDI-файлов в качестве входных данных 12
1.4. Выводы и результаты по главе 14
2. МЕТОДОЛОГИЯ 16
2.1. Music Transformer 16
2.2. Transcormer 18
2.3. Итеративная модель с двусторонним контекстом 19
2.4. Данные и параметры обучения 21
2.5. Выводы и результаты по главе 23
3. РЕЗУЛЬТАТЫ 24
3.1. Функция потерь и точность 24
3.2. Качественная оценка 25
3.3. Слушательский тест 30
3.4. Выводы и результаты по главе 32
ЗАКЛЮЧЕНИЕ 34
СПИСОК ЛИТЕРАТУРЫ 35

СПИСОК ЛИТЕРАТУРЫ
1. Chu, H., Urtasun, R., & Fidler, S. (2016). Song from PI: a musically plausible network for pop music generation. arXiv preprint arXiv:1611.03477.
2. Civit, M., Civit-Masot, J., Cuadrado, F. & Escalona, M. J. (2022). A systematic review of artificial intelligence-based music generation: Scope, applications and future trends. Expert Systems wi

In [35]:
def escape_latex(text):
    """
    Экранирует специальные
    символы LaTeX,
    сохраняя обычный текст.
    """

    if text is None:
        return ""

    text = str(text)

    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }

    result = []

    for char in text:

        result.append(
            replacements.get(
                char,
                char
            )
        )

    return "".join(
        result
    )

In [36]:
def parse_toc_entry(text):

    text = (
        text
        .strip()
    )

    match = re.match(
        r"^(.*?)(\d+)\s*$",
        text
    )

    if match:

        title = (
            match
            .group(1)
            .strip()
        )

        page_number = (
            match
            .group(2)
        )

    else:

        title = text
        page_number = ""

    title = re.sub(
        r"\.{2,}\s*$",
        "",
        title
    )

    title = re.sub(
        r"\s+",
        " ",
        title
    ).strip()

    return (
        title,
        page_number
    )

In [37]:
def latexify_text_with_urls(text):
    """
    Экранирует обычный текст для LaTeX,
    а URL оформляет через \\url{},
    чтобы LaTeX мог переносить их по строкам.
    """

    text = str(text)

    url_pattern = re.compile(
        r"https?://[^\s]+"
    )

    result = []
    last_end = 0

    for match in url_pattern.finditer(text):

        # Обычный текст до URL
        before = text[
            last_end:match.start()
        ]

        result.append(
            escape_latex(before)
        )

        url = match.group(0)

        # Если после URL случайно прилипла
        # обычная пунктуация, выносим её наружу.
        trailing = ""

        while (
            url
            and url[-1] in ".,;:"
        ):
            trailing = (
                url[-1]
                + trailing
            )
            url = url[:-1]

        result.append(
            r"\url{" + url + "}"
        )

        result.append(
            escape_latex(trailing)
        )

        last_end = match.end()

    # Остаток текста после последнего URL
    result.append(
        escape_latex(
            text[last_end:]
        )
    )

    return "".join(result)
def build_latex_document(
    toc_items,
    ref_items
):

    lines = []

    # ==================================================
    # PREAMBLE
    # ==================================================

    lines.append(
        r"\documentclass[14pt,a4paper]{extarticle}"
    )

    lines.append(
        r"\usepackage[a4paper,"
        r"left=30mm,"
        r"right=15mm,"
        r"top=20mm,"
        r"bottom=20mm]{geometry}"
    )

    lines.append(
        r"\usepackage{fontspec}"
    )

    lines.append(
        r"\usepackage{polyglossia}"
    )

    lines.append(
        r"\setmainlanguage{russian}"
    )

    lines.append(
        r"\setotherlanguage{english}"
    )

    # Если Times New Roman
    # установлен — используем его.
    # Иначе Times-совместимый
    # TeX Gyre Termes.
    lines.append(
        r"\IfFontExistsTF{Times New Roman}"
        r"{\setmainfont{Times New Roman}}"
        r"{\setmainfont{TeX Gyre Termes}}"
    )

    lines.append(
        r"\usepackage{tabularx}"
    )

    lines.append(
        r"\usepackage{array}"
    )

    lines.append(
        r"\usepackage{ragged2e}"
    )

    lines.append(
        r"\usepackage{enumitem}"
    )

    lines.append(
        r"\usepackage{xurl}"
    )

    lines.append(
        r"\setlength{\emergencystretch}{3em}"
    )

    lines.append(
        r"\usepackage{hyperref}"
    )

    lines.append(
        r"\hypersetup{"
        r"hidelinks,"
        r"breaklinks=true"
        r"}"
    )

    lines.append(
        r"\setlength{\parindent}{0pt}"
    )

    lines.append(
        r"\setlength{\parskip}{0pt}"
    )

    lines.append(
        r"\usepackage{setspace}"
        r"\onehalfspacing"
    )

    # ==================================================
    # DOC START
    # ==================================================

    lines.append(
        r"\begin{document}"
    )

    lines.append(
    r"\pagestyle{empty}"
    )

    # ==================================================
    # TOC HEADER
    # ==================================================

    lines.append(
        r"\begin{center}"
    )

    lines.append(
        r"\textbf{СОДЕРЖАНИЕ}"
    )

    lines.append(
        r"\end{center}"
    )

    lines.append(
        r"\vspace{0.5\baselineskip}"
    )

    # ==================================================
    # TOC
    #
    # X-колонка сама переносит
    # длинное название.
    #
    # \dotfill заполняет последнюю
    # строку точками.
    #
    # Правая колонка всегда
    # содержит номер страницы.
    # ==================================================

    lines.append(
        r"\begin{tabularx}"
        r"{\textwidth}"
        r"{@{}>{\raggedright\arraybackslash}X"
        r"@{}r@{}}"
    )

    for item in toc_items:

        title, page_number = (
            parse_toc_entry(
                item["text"]
            )
        )

        title = escape_latex(
            title
        )

        page_number = escape_latex(
            page_number
        )

        if page_number:

            lines.append(
                title
                + r"\dotfill"
                + " & "
                + page_number
                + r" \\"
            )

        else:

            lines.append(
                title
                + " & "
                + r" \\"
            )

    lines.append(
        r"\end{tabularx}"
    )

    # ==================================================
    # NEW PAGE
    # ==================================================

    lines.append(
        r"\newpage"
    )

    # ==================================================
    # REFERENCES HEADER
    # ==================================================

    lines.append(
        r"\begin{center}"
    )

    lines.append(
        r"\textbf{СПИСОК ЛИТЕРАТУРЫ}"
    )

    lines.append(
        r"\end{center}"
    )

    lines.append(
        r"\vspace{0.5\baselineskip}"
    )

    # ==================================================
    # REFERENCES
    # ==================================================

    lines.append(
        r"\begin{enumerate}"
        r"[label=\arabic*.,"
        r"leftmargin=0pt,"
        r"labelwidth=0pt,"
        r"labelsep=0.6em,"
        r"itemindent=0pt,"
        r"align=left,"
        r"itemsep=6pt,"
        r"parsep=0pt,"
        r"topsep=0pt]"
    )

    for entry in ref_items:

        entry = normalize_reference_text(
            entry
        )

        # Если исходная нумерация
        # каким-то образом сохранилась,
        # удаляем её.
        entry = re.sub(
            r"^\s*(?:"
            r"\[\d+\]"
            r"|\d+\."
            r"|\d+\)"
            r")\s*",
            "",
            entry
        )

        entry = latexify_text_with_urls(
            entry
        )

        lines.append(
            r"\item "
            + entry
        )

    lines.append(
        r"\end{enumerate}"
    )

    lines.append(
        r"\end{document}"
    )

    return "\n".join(
        lines
    )

In [38]:
latex_text = build_latex_document(
    toc_lines,
    ref_lines
)


with open(
    OUTPUT_TEX,
    "w",
    encoding="utf-8"
) as f:

    f.write(
        latex_text
    )


print(
    "\n✅ LaTeX-файл создан:"
)

print(
    OUTPUT_TEX
)

print(
    "\nTOC строк:",
    len(toc_lines)
)

print(
    "REFERENCE записей:",
    len(ref_lines)
)

print(
    "\nРазмер файла:",
    os.path.getsize(
        OUTPUT_TEX
    ),
    "байт"
)


✅ LaTeX-файл создан:
/content/drive/MyDrive/GOST_Project/smart_result_latex.tex

TOC строк: 20
REFERENCE записей: 23

Размер файла: 6961 байт
